# Financial Analysis & Business Insights
**Project 1 – Causal Inference & Synthetic Control for Algorithmic Pricing**
**Author:** Fatima  |  **Role:** Financial Analysis & Business Insights

This notebook covers my four tasks:
1. Understand pricing, sales, revenue and profitability metrics
2. Check the financial consistency of the data
3. Interpret the business impact of the pricing change
4. Produce tables and charts for the final pricing strategy report

**How to run:** open this file in VS Code, choose a Python kernel, then **Run All**.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.width", 160)

# --- find the data file (works from the notebooks/ folder or the repo root) ---
CSV = "algorithmic_pricing_synthetic_control_dataset.csv"
for p in [Path("../data") / CSV, Path("data") / CSV, Path(CSV)]:
    if p.exists():
        DATA_PATH = p
        break
else:
    raise FileNotFoundError("Put the CSV in the data/ folder of the repo.")

# --- where to save charts and tables ---
if DATA_PATH.parent.name == "data":
    REPORTS = DATA_PATH.parent.parent / "reports"
else:
    REPORTS = Path("reports")
FIG_DIR = REPORTS / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
TREATMENT_START = df.loc[df["post_period"] == 1, "date"].min()
print("Rows, columns:", df.shape)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Pricing change starts:", TREATMENT_START.date())
df.head()

## Task 1 – Understand the metrics

### Data dictionary
| Column | Meaning |
|---|---|
| date | Day of the observation |
| market | Market_A to Market_E |
| product_id | P001, P002, P003 |
| price | Selling price per unit |
| units_sold | Units sold that day |
| unit_cost | Cost to the business per unit |
| competitor_price | Competitor's price for the same product |
| promotion | 1 if a promotion ran, 0 otherwise |
| treatment_market | 1 for the market that received the new pricing (Market_A) |
| post_period | 1 for dates on or after the pricing change |
| treated_post | treatment_market × post_period (1 = treated market after the change) |
| revenue | price × units_sold |
| total_cost | unit_cost × units_sold |
| profit | revenue − total_cost |

### Metrics used
- **Revenue** = price × units sold
- **Profit** = revenue − total cost
- **Profit margin %** = profit ÷ revenue × 100
- **Unit margin** = price − unit cost
- **Price gap vs competitor** = price − competitor price (positive means we are more expensive)

In [ ]:
print("Markets :", sorted(df["market"].unique()))
print("Products:", sorted(df["product_id"].unique()))
print("Treated market(s):", df.loc[df["treatment_market"] == 1, "market"].unique().tolist())
print("Days:", df["date"].nunique(), "| Rows per market:", len(df) // df["market"].nunique())
df.describe().T

## Task 2 – Financial consistency checks
These checks confirm the numbers add up before anyone builds models on them.
A tolerance of 0.50 is allowed because prices are rounded to 2 decimals.

In [ ]:
TOL = 0.50
expected_rows = df["date"].nunique() * df["market"].nunique() * df["product_id"].nunique()
checks = {
    "Missing values (total)":                   int(df.isna().sum().sum()),
    "Duplicate date-market-product rows":       int(df.duplicated(["date", "market", "product_id"]).sum()),
    "Rows where price < unit cost":             int((df["price"] < df["unit_cost"]).sum()),
    "Rows with zero/negative units":            int((df["units_sold"] <= 0).sum()),
    "Rows with zero/negative price":            int((df["price"] <= 0).sum()),
    "Revenue != price x units (> tolerance)":   int(((df["price"] * df["units_sold"] - df["revenue"]).abs() > TOL).sum()),
    "Total cost != unit cost x units (> tol.)": int(((df["unit_cost"] * df["units_sold"] - df["total_cost"]).abs() > TOL).sum()),
    "Profit != revenue - total cost (> 0.01)":  int(((df["revenue"] - df["total_cost"] - df["profit"]).abs() > 0.01).sum()),
    "treated_post flag wrong":                  int((df["treatment_market"] * df["post_period"] != df["treated_post"]).sum()),
    "Missing rows in the full panel":           int(expected_rows - len(df)),
}
check_table = pd.DataFrame({"Issues found": checks})
check_table["Result"] = np.where(check_table["Issues found"] == 0, "PASS", "REVIEW")
display(check_table)

print("Largest revenue rounding difference :", round((df.price * df.units_sold - df.revenue).abs().max(), 3))
print("Largest cost rounding difference    :", round((df.unit_cost * df.units_sold - df.total_cost).abs().max(), 3))

In [ ]:
# Does the unit cost stay stable? (profit changes should not come from cost changes)
cost_check = df.groupby(["treatment_market", "post_period"])["unit_cost"].mean().unstack()
cost_check.index = ["Other markets", "Market_A (treated)"]
cost_check.columns = ["Before", "After"]
display(cost_check)

# Is the treatment date clean? (the flag should switch on at one date only)
print("Pre-period ends:", df.loc[df.post_period == 0, "date"].max().date(),
      "| Post-period starts:", df.loc[df.post_period == 1, "date"].min().date())

**Conclusion of the consistency checks:** the dataset is complete and financially consistent.
Revenue, cost and profit follow the expected formulas. The only differences are tiny rounding
differences (under 0.50 per row), which do not affect any conclusions. Unit costs did not change
between the periods, so any profit change comes from pricing and volume, not from costs.

## Task 1 (continued) – Pricing, sales, revenue and profit metrics

In [ ]:
df["margin_pct"]     = df["profit"] / df["revenue"] * 100
df["unit_margin"]    = df["price"] - df["unit_cost"]
df["price_gap_comp"] = df["price"] - df["competitor_price"]
df["period"]         = np.where(df["post_period"] == 1, "After", "Before")
df["group"]          = np.where(df["treatment_market"] == 1, "Market_A (treated)", "Other markets")

metrics = ["price", "units_sold", "revenue", "total_cost", "profit", "margin_pct", "unit_margin", "price_gap_comp"]

print("Average per market-product-day, by market")
display(df.groupby("market")[metrics].mean())

print("Average per market-product-day, by product")
display(df.groupby("product_id")[metrics].mean())

print("Totals by market (whole period)")
display(df.groupby("market")[["units_sold", "revenue", "total_cost", "profit"]].sum())

## Task 3 – Impact of the pricing change
We compare the treated market (Market_A) with the other markets, before and after the change.
The **difference-in-differences (DiD)** estimate is:

(Market_A after − Market_A before) − (Other markets after − Other markets before)

It removes changes that happened everywhere (such as seasonality), leaving the change linked to the new pricing.
This is a quick first estimate. The formal estimate comes from the Synthetic Control and DML work.

In [ ]:
order = ["Other markets", "Market_A (treated)"]
before_after = df.groupby(["group", "period"])[metrics].mean()
display(before_after.loc[order])

def did(frame, col):
    g = frame.groupby(["treatment_market", "post_period"])[col].mean().unstack()
    return (g.loc[1, 1] - g.loc[1, 0]) - (g.loc[0, 1] - g.loc[0, 0])

rows = []
for c in metrics + ["competitor_price", "promotion"]:
    g = df.groupby(["treatment_market", "post_period"])[c].mean().unstack()
    rows.append({
        "metric": c,
        "Market_A before": g.loc[1, 0], "Market_A after": g.loc[1, 1],
        "Others before": g.loc[0, 0], "Others after": g.loc[0, 1],
        "DiD effect": did(df, c),
    })
did_table = pd.DataFrame(rows).set_index("metric")
display(did_table)

In [ ]:
# Percentage view, easy to quote in the report
cols = ["price", "units_sold", "revenue", "profit", "margin_pct"]
a = df[df.treatment_market == 1].groupby("post_period")[cols].mean()
o = df[df.treatment_market == 0].groupby("post_period")[cols].mean()
pct_table = pd.DataFrame({"Market_A % change": (a.loc[1] / a.loc[0] - 1) * 100,
                          "Other markets % change": (o.loc[1] / o.loc[0] - 1) * 100})
display(pct_table)

### Charts – profit, price, units and margin over time (treated vs other markets)

In [ ]:
def trend_chart(col, title, ylabel, fname):
    daily = df.groupby(["date", "group"])[col].mean().unstack()[order].rolling(7).mean()
    fig, ax = plt.subplots(figsize=(10, 5))
    daily.plot(ax=ax, linewidth=2)
    ax.axvline(TREATMENT_START, color="red", linestyle="--", label="Pricing change")
    ax.set_title(title); ax.set_ylabel(ylabel); ax.set_xlabel("")
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(FIG_DIR / fname, dpi=150)
    plt.show()

trend_chart("profit",     "Average daily profit per product (7-day average)", "Profit",     "profit_trend.png")
trend_chart("price",      "Average selling price (7-day average)",            "Price",      "price_trend.png")
trend_chart("units_sold", "Average daily units sold (7-day average)",         "Units sold", "units_trend.png")
trend_chart("margin_pct", "Profit margin % (7-day average)",                  "Margin %",   "margin_trend.png")

### Does the effect differ by product?

In [ ]:
prod_rows = []
for p in sorted(df.product_id.unique()):
    s = df[df.product_id == p]
    prod_rows.append({"product": p,
                      "price effect":   did(s, "price"),
                      "units effect":   did(s, "units_sold"),
                      "revenue effect": did(s, "revenue"),
                      "profit effect":  did(s, "profit")})
prod_table = pd.DataFrame(prod_rows).set_index("product")
display(prod_table)

ax = prod_table[["revenue effect", "profit effect"]].plot(kind="bar", figsize=(8, 5))
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Estimated effect of the pricing change by product (DiD, per day)")
ax.set_ylabel("Change vs other markets"); ax.set_xlabel("")
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(FIG_DIR / "effect_by_product.png", dpi=150); plt.show()

### Possible confounders
Other things may have changed at the same time as the price. If they differ between Market_A and the
other markets, they could explain part of the result: competitor prices, promotions and unit cost.

In [ ]:
conf = df.groupby(["group", "period"])[["competitor_price", "promotion", "unit_cost", "price_gap_comp"]].mean().loc[order]
display(conf)

### Is the effect larger than normal noise? (placebo test)
We pretend each *untreated* market got the pricing change and compute the same DiD against the
remaining untreated markets. If the real Market_A effect is far bigger than these placebo values, it is
unlikely to be chance.

In [ ]:
controls = df[df.market != "Market_A"].copy()
placebo = {}
for m in sorted(controls.market.unique()):
    tmp = controls.copy()
    tmp["treatment_market"] = (tmp.market == m).astype(int)
    placebo[m] = did(tmp, "profit")

actual = did(df, "profit")
placebo_table = pd.Series({**placebo, "Market_A (actual)": actual}, name="profit DiD")
display(placebo_table.to_frame())

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["grey"] * len(placebo) + ["tab:red"]
ax.bar(placebo_table.index, placebo_table.values, color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Profit effect: real treated market vs placebo markets")
ax.set_ylabel("Profit DiD per day")
plt.xticks(rotation=20); plt.tight_layout()
plt.savefig(FIG_DIR / "placebo_test.png", dpi=150); plt.show()

In [ ]:
# Pre-trend check: was Market_A moving in parallel with the other markets before April?
monthly = df.groupby([df.date.dt.to_period("M"), "group"])["profit"].mean().unstack()[order]
monthly["gap (A - others)"] = monthly["Market_A (treated)"] - monthly["Other markets"]
display(monthly)

### Business size of the effect

In [ ]:
a_post = df[(df.treatment_market == 1) & (df.post_period == 1)]
n_rows = len(a_post)
incremental_profit  = did(df, "profit")  * n_rows
incremental_revenue = did(df, "revenue") * n_rows
actual_profit_post  = a_post["profit"].sum()

print(f"Market_A post-period rows (product-days)      : {n_rows}")
print(f"Estimated extra profit over the post period   : {incremental_profit:,.0f}")
print(f"Estimated revenue change over the post period : {incremental_revenue:,.0f}")
print(f"Market_A actual profit in the post period     : {actual_profit_post:,.0f}")
print(f"Extra profit as % of actual profit            : {incremental_profit / actual_profit_post * 100:.1f}%")

In [ ]:
# Save the tables so they can be pasted into the final report
check_table.to_csv(REPORTS / "table_consistency_checks.csv")
did_table.to_csv(REPORTS / "table_did_effects.csv")
prod_table.to_csv(REPORTS / "table_effect_by_product.csv")
before_after.loc[order].to_csv(REPORTS / "table_before_after.csv")
print("Saved tables and figures in:", REPORTS.resolve())

## Task 4 – Findings for the final report

**Summary**
- The data passes all financial consistency checks. Only tiny rounding differences exist.
- Market_A raised its price by about 8% on 1 April 2024. Other markets kept their prices flat.
- Units sold in Market_A fell by about 3%, while other markets grew by about 6%.
- Revenue stayed roughly flat compared with other markets, but costs fell because fewer units were sold.
- Profit rose compared with other markets (DiD about +57 per product-day) and margin improved by about 4 percentage points.
- The profit effect is much larger than the placebo effects in untreated markets.

**Caveats to mention**
- Market_A already had higher prices and higher margins before the change.
- Market_A ran fewer promotions after the change, which may add to the effect.
- Market_A moved from about 2 above the competitor's price to about 5 above, so long-run customer loss is a risk.
- This is a quick DiD estimate with one treated market. The Synthetic Control and DML results should confirm or refine it.

Full write-up: `reports/financial_analysis_fatima.md`.